In [ ]:
from experiment import run_experiment

def main():
    """
    메인 실행 함수. 실험을 수행하고 결과를 출력.
    """
    author_file = "./gsc_data/authors.all"
    data_folder = "./gsc_data/DATA/"
    top_n = 20  # 비교할 상위 저자 수
    current_year = 2013  # 최신 연도 설정
    t = 5  # Timed h-index에서 사용할 최근 연도 수
    gamma = 4  # contemporary_h_index 파라미터
    delta = 1  # contemporary_h_index 파라미터
    a, b = 50, 50  # expert_index 파라미터
    
    print("Running experiment...")
    result_df = run_experiment(author_file, data_folder, top_n, current_year, t, gamma, delta, a, b)
    
    print("\nComparison Matrix:")
    print(result_df)
    
    result_df.to_csv("./results/comparison_matrix.csv", index=True)
    print("\nExperiment results saved to ./results/comparison_matrix.csv")
    print("Index statistics saved to ./results/index_statistics.csv")

if __name__ == "__main__":
    main()


In [ ]:
from data_loader import load_authors, load_papers
from index_calculator import (
    h_index, g_index, i10_index, ar_index, contemporary_h_index, ha_index,
    career_year_h_index, timed_h_index, expert_index, preprocess_expert_index
)
from index_comparator import compare_indices
import pandas as pd

In [ ]:
def preprocess_global_expert_index(author_files, data_folder, current_year):
    """
    전체 연구자의 품질 및 최신성 최댓값을 구하는 전처리 과정.
    """
    global max_quality, max_combined_recency

    authors_df = load_authors(author_files)
    papers_cache = {}
    all_papers = []
    
    for _, row in authors_df.iterrows():
        author_id = row['ID']
        if author_id not in papers_cache:
            papers_cache[author_id] = load_papers(data_folder, author_id)
        all_papers.extend(papers_cache[author_id])

    if not all_papers:
        raise ValueError("No papers found. Ensure that the dataset is correctly loaded.")

    preprocess_expert_index(all_papers, current_year)

In [ ]:
author_file = "./gsc_data/authors.all"
data_folder = "./gsc_data/DATA/"
top_n = 20  # 비교할 상위 저자 수
current_year = 2013  # 최신 연도 설정
t = 5  # Timed h-index에서 사용할 최근 연도 수
gamma = 4  # contemporary_h_index 파라미터
delta = 1  # contemporary_h_index 파라미터
a, b = 50, 50  # expert_index 파라미터


authors_df = load_authors(author_file)
papers_cache = {}
rankings = {
    "h_index": [], "g_index": [], "i10_index": [], "ar_index": [], "contemporary_h_index": [],
    "ha_index": [], "career_year_h_index": [], "timed_h_index": [], "expert_index": []
}

print("Preprocessing expert index...")  # 🔹 전처리 시작 메시지 추가
preprocess_global_expert_index(author_file, data_folder, current_year)
print("Preprocessing complete. Running experiment...")  # 🔹 전처리 완료 메시지 추가

for _, row in authors_df.iterrows():
    author_id = row['ID']
    if author_id not in papers_cache:
        papers_cache[author_id] = load_papers(data_folder, author_id)
    papers = papers_cache[author_id]

    rankings["h_index"].append((author_id, h_index(papers)))
    rankings["g_index"].append((author_id, g_index(papers)))
    rankings["i10_index"].append((author_id, i10_index(papers)))
    rankings["ar_index"].append((author_id, ar_index(papers, current_year)))
    rankings["contemporary_h_index"].append((author_id, contemporary_h_index(papers, current_year, gamma, delta)))
    rankings["ha_index"].append((author_id, ha_index(papers, current_year)))
    rankings["career_year_h_index"].append((author_id, career_year_h_index(papers, current_year)))
    rankings["timed_h_index"].append((author_id, timed_h_index(papers, current_year, t)))
    rankings["expert_index"].append((author_id, expert_index(papers, current_year, a, b)))

for key in rankings:
    rankings[key] = sorted(rankings[key], key=lambda x: x[1], reverse=True)

# 🔹 compare_indices() 호출 전 rankings 변환
rankings_ids = {key: [author_id for author_id, _ in rankings[key]] for key in rankings}
comparison_matrix = compare_indices(rankings_ids, top_n)

df_comparison = pd.DataFrame(comparison_matrix)

In [ ]:
print(authors_df)

In [ ]:
print(papers_cache['LrG_vOwAAAAJ'])

In [ ]:
print(rankings['h_index'])